# Agents, MCP & Skills

En praktisk workshop hvor vi udforsker den nye verden af **agentic software engineering**.

---

## Agenda (ca. 3,5 timer)

| Tid | Emne | Øvelser |
|-----|------|----------|
| 20 min | Intro, Recap & Jupyter Notebooks med Swift | |
| 70 min | Agenter - teori og byg din egen | **Øvelse 1:** Tool Execute-funktioner, **Øvelse 2:** Human-in-the-loop |
| 20 min | *Pause* | |
| 45 min | Abstraktionsstigen – MCP | **Øvelse 3:** Protocol-based Tools |
| 20 min | Extension: get_nearest_station | **Øvelse 4:** Nyt tool |
| 25 min | Skills – teori og prøv med GitHub Copilot | SKILL.md |
| 10 min | Afrunding | |

---

# Del 0: Intro, Recap & Jupyter Notebooks med Swift
*20 minutter*

#### ℹ️ Installér Ollama nu, da det tager lidt tid

**Windows (PowerShell):**
```powershell
winget install Ollama.Ollama
```

**Mac (Homebrew):**
```bash
brew install ollama
```

Start Ollama og hiv modellen ned:
```bash
ollama serve
ollama pull llama3.1:8b
```

👆🏻 kan også foretages i GUI; vælg model, skriv en prompt (f.eks. "hej") og tryk retur

## Opsummering fra sidste gang

- LLMs & Agents - What are they, and why can they code? — Martin
- Agentisk softwareudvikling for personlig produktivitet — Emil
- Agentisk Software Engineering for fuld SDLC af et produkt — Magnus

Lige denne planche er meget relevant mht. dagens seance: [How LLMs Learn: The Three Stages](https://martinrl.github.io/presentations/what-are-llms-and-agents/index.html#14)

<img src="images/LLMs-14.png" alt="LLMs-14" width="600" style="max-width: 100%;">

---

## Intro til Jupyter Notebooks med Swift

### Hvad er en Jupyter Notebook med Swift?

- `.ipynb` format (JSON-baseret)
- Kører på Swift Jupyter kernel (swift-jupyter)
- Understøtter Swift med fuld adgang til Foundation og andre frameworks
- Velegnet til prototyping og eksperimenter

### Sådan kører du en celle

- **Shift+Enter** - Kør celle og gå til næste
- **Ctrl+Enter** - Kør celle og bliv
- Variabler persisterer på tværs af celler

### Opsætning

```bash
# Installér Swift Jupyter kernel (kræver Swift toolchain)
# Se: https://github.com/google/swift-jupyter
pip install swift-jupyter
```

In [ ]:
print("Hello EV World!")

---

## Det Agentiske Landskab

<img src="images/Orloff-1600x1067.webp" alt="Professor Orloff" width="600" style="max-width: 100%;">
<br><br>

> An LLM is just a brain in a jar. 🧠

— [Simon Maple](https://www.linkedin.com/in/simonmaple/), Head of Developer Relations @ Tessl ([AI Native podcast](https://open.spotify.com/episode/1HYcUlgkCa9VwvVfuy3fyx?nd=1&dlsi=a77f69fabf8b46b1))

### Hvorfor kan LLM'er kode så godt nu?

**Kode har indbygget feedback.** Til forskel fra "god tekst" kan vi objektivt måle om kode virker:
- Kompilerer det? ✅/❌
- Består tests? ✅/❌
- Løser det opgaven? ✅/❌

Dette muliggør **RLVR** (*Reinforcement Learning with Verifiable Rewards*):

<img src="images/ai-training-loop.svg" alt="AI Træningsloop" width="600" style="max-width: 100%;">

### Benchmarks (pr. januar 2026)

#### SWE-bench: Løs rigtige GitHub issues

**SWE-bench Verified** (500 human-validerede issues):

| # | Model | Score |
|---|-------|-------|
| 1 | Claude Opus 4.5 | 80.9% |
| 2 | GPT-5.2 | 75.4% |
| 3 | Claude Opus 4.1 | 74.5% |

**SWE-bench Pro** (1865 sværere issues, nye repos):

| # | Agent | Score |
|---|-------|-------|
| 1 | Claude Opus 4.5 | 45.9% |
| 2 | Claude Sonnet 4.5 | 43.6% |
| 3 | Gemini 3 Pro Preview | 43.3% |

[swebench.com](https://www.swebench.com/)

---

#### METR: Hvor lang tid kan AI arbejde selvstændigt?

METR måler **tidshorisonten** - hvor lange opgaver kan AI løse med 50% succes?

| # | Model | 50% Tidshorisont |
|---|-------|------------------|
| 1 | Claude Opus 4.5 | ~4.8 timer |
| 2 | GPT-5.1 Codex Max | ~2.9 timer |
| 3 | GPT-5 | ~2.3 timer |

<br><img src="images/metr-timeline.svg" alt="METR Tidslinje" width="600" style="max-width: 100%;">

**Bemærk:**
- Tidshorisonten **fordobles hver 7. måned** (konsistent over 6 år, dvs. en pendant til Moore's Law)

**Hvis trenden fortsætter:**
- 2-4 år: ugelange opgaver
- Årtiet ud: månedlange projekter

[metr.org](https://metr.org/blog/2025-03-19-measuring-ai-ability-to-complete-long-tasks/)

---

#### DPAI Arena: Full developer workflow

JetBrains' benchmark - evaluerer hele engineering lifecycle på Spring-projekter (140+ opgaver).

| # | Agent | Model | Score |
|---|-------|-------|-------|
| 1 | Junie CLI 533.2 | Claude Opus 4.5 | 68.9 |
| 2 | Claude Code 2.0.55 | Claude Opus 4.5 | 68.0 |
| 3 | Junie CLI 496.3-prototype | Claude Sonnet 4.5 | 68.0 |

[dpaia.dev](https://dpaia.dev/)

---

### Men pas på: Benchmark ≠ virkelighed

METR-studie (juli 2025):
- Erfarne udviklere *troede* de var **20% hurtigere** med AI, når objektive tests viste de var **19% langsommere**

Sikkerhedssårbarheder ([Stanford/Boneh 2023](https://arxiv.org/abs/2211.03622)):
- Mennesker uden AI: **~50%** sårbar kode
- Mennesker MED AI: **~64%** — faktisk **værre**!

Forskning tager lang tid og der innoveres med lynets hast, så det handler i høj grad om at etablere et væddemål.

## Prerequisites Verification

### 1. Er Ollama installeret og kørende?

In [ ]:
import Foundation

let selectedModelName = "llama3.1:8b"

let url = URL(string: "http://localhost:11434/api/tags")!
do {
    let (data, _) = try await URLSession.shared.data(from: url)
    print("✅ Ollama kører! Model: \(selectedModelName)")
} catch {
    print("❌ Ollama kører IKKE. Start med: ollama serve")
}

---

# Del 1: Agenter - teori og byg din egen
*70 minutter inkl. øvelser*

> **"The LLM never actually touches your filesystem. It just *asks* for things to happen."**
> — Mihail Eric, ["The Emperor Has No Clothes"](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)

AI coding assistants virker som magi, men kernen er simpel:
- **~200 linjer kode** er alt der skal til
- LLM beslutter hvad der skal ske
- **Din kode** udfører handlingerne lokalt
- Resultater sendes tilbage til LLM

### The Agentic Loop - 4 Steps

<img src="images/agentic-loop.svg" alt="Agentic Loop" width="600" style="max-width: 100%;">

**Nøgleindsigt:** LLM'en kalder aldrig noget. Den outputter JSON der siger "Jeg vil gerne kalde X med Y". **Vi** eksekverer det!

## 1.1 Hvad LLM'en Faktisk Ser

Lad os se præcis hvad der sendes til Ollama.

In [ ]:
// Dette er PRÆCIS hvad vi sender til Ollama som tool definition. Det er bare JSON der beskriver en funktion.

let toolJsonExample = """
{
  "type": "function",
  "function": {
    "name": "get_station_status",
    "description": "Get real-time status of an EV charging station",
    "parameters": {
      "type": "object",
      "properties": {
        "stationId": {
          "type": "string",
          "description": "The station ID, e.g., CPH-001"
        }
      },
      "required": ["stationId"]
    }
  }
}
"""

print("Dette er hvad LLM'en ser som tool definition:")
print(toolJsonExample)

In [ ]:
// Og dette er hvad LLM'en returnerer når den vil bruge et tool

let toolCallResponseExample = """
{
  "message": {
    "role": "assistant",
    "content": "",
    "tool_calls": [
      {
        "function": {
          "name": "get_station_status",
          "arguments": {
            "stationId": "CPH-001"
          }
        }
      }
    ]
  }
}
"""

print("Når LLM'en vil kalde et tool, returnerer den:")
print(toolCallResponseExample)
print("\nLLM'en eksekverer ikke noget - den beder os om at gøre det!")

## 1.2 SimpleTool - Den Transparente Abstraktion

Nu bygger vi vores egen tool-definition.

**Forskellen på SimpleTool og Ollama's REST API:**
- `SimpleTool` har en `execute` closure - du kan se præcis hvad der kører
- Ollama's API er kun en JSON-struktur, dvs. "automagi"

In [ ]:
import Foundation

struct SimpleTool {
    let name: String
    let description: String
    let parameterSchema: [String: Any]
    let execute: ([String: Any]) -> String
}

print("✅ SimpleTool defineret!")

## 1.3 ChargeSmart Domæne - Mock Data

Vi arbejder med et fiktivt EV-ladenetværk i København:

In [ ]:
// Station data (med koordinater til øvelse 4: get_nearest_station)
struct StationInfo {
    let id: String
    let name: String
    let location: String
    let powerKw: Int
    let type: String
    let latitude: Double
    let longitude: Double
}

struct SessionInfo {
    let vehicleId: String
    let start: Date
    let kwh: Double
}

let stations: [String: StationInfo] = [
    "CPH-001": StationInfo(id: "CPH-001", name: "Nørreport Station", location: "Nørreport", powerKw: 150, type: "ultra-fast", latitude: 55.6839, longitude: 12.5715),
    "CPH-002": StationInfo(id: "CPH-002", name: "Fisketorvet Parking", location: "Fisketorvet", powerKw: 50, type: "fast", latitude: 55.6692, longitude: 12.5519),
    "CPH-003": StationInfo(id: "CPH-003", name: "Tivoli Garage", location: "Tivoli", powerKw: 50, type: "fast", latitude: 55.6736, longitude: 12.5681),
    "CPH-004": StationInfo(id: "CPH-004", name: "Ørestad Center", location: "Ørestad", powerKw: 150, type: "ultra-fast", latitude: 55.6310, longitude: 12.5770),
    "CPH-005": StationInfo(id: "CPH-005", name: "Amager Strandpark", location: "Amager", powerKw: 22, type: "slow", latitude: 55.6520, longitude: 12.6050),
    "CPH-006": StationInfo(id: "CPH-006", name: "Nørrebro Runddel", location: "Nørrebro", powerKw: 50, type: "fast", latitude: 55.7015, longitude: 12.5450),
    "CPH-007": StationInfo(id: "CPH-007", name: "Frederiksberg Have", location: "Frederiksberg", powerKw: 7, type: "slow", latitude: 55.6784, longitude: 12.5293),
    "CPH-008": StationInfo(id: "CPH-008", name: "Kastrup Lufthavn P4", location: "Kastrup", powerKw: 150, type: "ultra-fast", latitude: 55.6180, longitude: 12.6560),
    "CPH-009": StationInfo(id: "CPH-009", name: "Valby Langgade", location: "Valby", powerKw: 22, type: "slow", latitude: 55.6631, longitude: 12.5120),
    "CPH-010": StationInfo(id: "CPH-010", name: "Hellerup Station", location: "Hellerup", powerKw: 50, type: "fast", latitude: 55.7280, longitude: 12.5720),
]

// Simuler aktive sessioner
let activeSessions: [String: SessionInfo] = [
    "CPH-002": SessionInfo(vehicleId: "Tesla-Model-3", start: Date().addingTimeInterval(-45 * 60), kwh: 28.5),
    "CPH-004": SessionInfo(vehicleId: "Polestar-2", start: Date().addingTimeInterval(-12 * 60), kwh: 18.2),
    "CPH-007": SessionInfo(vehicleId: "VW-ID4", start: Date().addingTimeInterval(-120 * 60), kwh: 11.0),
]

// Tarif struktur
func getTariff(hour: Int) -> Double {
    switch hour {
    case 0..<6: return 1.50   // Off-peak: 00:00-06:00
    case 6..<17: return 2.50  // Normal: 06:00-17:00
    case 17..<21: return 4.00 // Peak: 17:00-21:00
    default: return 2.50      // Normal: 21:00-00:00
    }
}

print("✅ Loaded \(stations.count) stationer, \(activeSessions.count) aktive sessioner")

## 1.4 Øvelse 1: Forstå Execute-funktionerne til ChargeSmart Tools
*~15 minutter*

Nu skal du **forstå og evt. modificere** `execute`-closures for 3 ChargeSmart tools.

**📋 Opgave:** Før du kører cellen nedenfor:
1. Læs koden igennem og forstå hvad hver closure gør
2. Prøv at skjule koden og skrive din egen version (brug hints nedenfor)
3. Sammenlign med løsningen og kør cellen

**Du har adgang til:**
- `stations` - Dictionary med alle stationer (ID → StationInfo)
- `activeSessions` - Dictionary med aktive sessioner (stationId → session info)
- `getTariff(hour:)` - Returnerer pris pr. kWh for en given time

**Krav til de 3 tools:**

| Tool | Input | Output |
|------|-------|--------|
| `get_station_status` | stationId (string) | Status på én station (ledig/optaget, kW, type) |
| `list_stations` | *ingen* | Liste over alle stationer med status |
| `calculate_charging_cost` | kwh (number), hour (int) | Pris for at lade X kWh på tidspunkt Y |

In [ ]:
// 🎯 ØVELSE 1: Studer disse Execute-closures!
// Tip: args er [String: Any] - brug args["stationId"] as? String osv.

var tools: [String: SimpleTool] = [
    "get_station_status": SimpleTool(
        name: "get_station_status",
        description: "Get real-time status of an EV charging station including availability, power level, and current session info",
        parameterSchema: [
            "type": "object",
            "properties": ["stationId": ["type": "string", "description": "Station ID like CPH-001"]],
            "required": ["stationId"]
        ],
        execute: { args in
            // 1. Hent stationId fra args
            guard let stationId = args["stationId"] as? String else { return "Error: Missing stationId" }

            // 2. Slå op i stations dictionary
            guard let station = stations[stationId] else { return "Error: Station \(stationId) not found" }

            // 3. Tjek om der er en aktiv session
            if let session = activeSessions[stationId] {
                let formatter = DateFormatter()
                formatter.dateFormat = "HH:mm"
                return "Station \(stationId) (\(station.name)): IN USE by \(session.vehicleId) since \(formatter.string(from: session.start)), \(String(format: "%.1f", session.kwh)) kWh delivered. \(station.powerKw)kW \(station.type) charger."
            }

            // 4. Returner beskrivende streng
            return "Station \(stationId) (\(station.name)): AVAILABLE. \(station.powerKw)kW \(station.type) charger at \(station.location)."
        }
    ),

    "list_stations": SimpleTool(
        name: "list_stations",
        description: "List all EV charging stations in the ChargeSmart network with their current availability",
        parameterSchema: ["type": "object", "properties": [:] as [String: Any]],
        execute: { _ in
            let lines = stations.values.sorted(by: { $0.id < $1.id }).map { s in
                let status = activeSessions.keys.contains(s.id) ? "IN USE" : "AVAILABLE"
                return "- \(s.id): \(s.name) (\(s.location)) - \(s.powerKw)kW \(s.type) - \(status)"
            }
            return "ChargeSmart Network Stations:\n" + lines.joined(separator: "\n")
        }
    ),

    "calculate_charging_cost": SimpleTool(
        name: "calculate_charging_cost",
        description: "Calculate the cost of charging based on kWh needed and time of day",
        parameterSchema: [
            "type": "object",
            "properties": [
                "kwh": ["type": "number", "description": "Energy needed in kWh"],
                "hour": ["type": "integer", "description": "Hour of day (0-23)"]
            ],
            "required": ["kwh", "hour"]
        ],
        execute: { args in
            // Handle both number and string values from LLM
            let kwh: Double
            if let kwhNum = args["kwh"] as? Double {
                kwh = kwhNum
            } else if let kwhInt = args["kwh"] as? Int {
                kwh = Double(kwhInt)
            } else if let kwhStr = args["kwh"] as? String, let parsed = Double(kwhStr) {
                kwh = parsed
            } else {
                return "Error: Invalid kwh value"
            }

            let hour: Int
            if let hourInt = args["hour"] as? Int {
                hour = hourInt
            } else if let hourDbl = args["hour"] as? Double {
                hour = Int(hourDbl)
            } else if let hourStr = args["hour"] as? String, let parsed = Int(hourStr) {
                hour = parsed
            } else {
                return "Error: Invalid hour value"
            }

            let tariff = getTariff(hour: hour)
            let cost = kwh * tariff

            let period: String
            switch hour {
            case 0..<6: period = "off-peak"
            case 17..<21: period = "peak"
            default: period = "normal"
            }

            return "Charging \(String(format: "%.1f", kwh)) kWh at \(String(format: "%02d", hour)):00 (\(period) tariff: \(String(format: "%.2f", tariff)) DKK/kWh) = \(String(format: "%.2f", cost)) DKK"
        }
    ),
]

print("✅ Defineret \(tools.count) tools:")
for tool in tools.values {
    let desc = String(tool.description.prefix(50))
    print("   - \(tool.name): \(desc)...")
}

### Hints til Øvelse 1

<details>
<summary>💡 Hint 1: Sådan læser du fra args dictionary</summary>

```swift
// Hent en string property
guard let stationId = args["stationId"] as? String else { return "Error" }

// Hent et tal (kan være Int, Double eller String fra JSON)
let kwh: Double
if let kwhNum = args["kwh"] as? Double {
    kwh = kwhNum
} else if let kwhStr = args["kwh"] as? String, let parsed = Double(kwhStr) {
    kwh = parsed
} else {
    return "Error: Invalid kwh"
}
```

</details>

<details>
<summary>💡 Hint 2: get_station_status struktur</summary>

```swift
execute: { args in
    guard let stationId = args["stationId"] as? String else { return "Error" }
    guard let station = stations[stationId] else { return "Error: not found" }

    if let session = activeSessions[stationId] {
        // Returner "IN USE" besked
    }
    // Returner "AVAILABLE" besked
}
```

</details>

<details>
<summary>💡 Hint 3: list_stations med sorted/map</summary>

```swift
execute: { _ in
    let lines = stations.values.sorted(by: { $0.id < $1.id }).map { s in
        let status = activeSessions.keys.contains(s.id) ? "IN USE" : "AVAILABLE"
        return "- \(s.id): \(s.name) - \(status)"
    }
    return lines.joined(separator: "\n")
}
```

</details>

<details>
<summary>✅ Fuld løsning: Alle 3 tools</summary>

```swift
var tools: [String: SimpleTool] = [
    "get_station_status": SimpleTool(
        name: "get_station_status",
        description: "Get real-time status of an EV charging station including availability, power level, and current session info",
        parameterSchema: [
            "type": "object",
            "properties": ["stationId": ["type": "string", "description": "Station ID like CPH-001"]],
            "required": ["stationId"]
        ],
        execute: { args in
            guard let stationId = args["stationId"] as? String else { return "Error: Missing stationId" }
            guard let station = stations[stationId] else { return "Error: Station \(stationId) not found" }
            if let session = activeSessions[stationId] {
                let formatter = DateFormatter()
                formatter.dateFormat = "HH:mm"
                return "Station \(stationId) (\(station.name)): IN USE by \(session.vehicleId) since \(formatter.string(from: session.start)), \(String(format: "%.1f", session.kwh)) kWh delivered. \(station.powerKw)kW \(station.type) charger."
            }
            return "Station \(stationId) (\(station.name)): AVAILABLE. \(station.powerKw)kW \(station.type) charger at \(station.location)."
        }
    ),
    "list_stations": SimpleTool(
        name: "list_stations",
        description: "List all EV charging stations in the ChargeSmart network with their current availability",
        parameterSchema: ["type": "object", "properties": [:] as [String: Any]],
        execute: { _ in
            let lines = stations.values.sorted(by: { $0.id < $1.id }).map { s in
                let status = activeSessions.keys.contains(s.id) ? "IN USE" : "AVAILABLE"
                return "- \(s.id): \(s.name) (\(s.location)) - \(s.powerKw)kW \(s.type) - \(status)"
            }
            return "ChargeSmart Network Stations:\n" + lines.joined(separator: "\n")
        }
    ),
    "calculate_charging_cost": SimpleTool(
        name: "calculate_charging_cost",
        description: "Calculate the cost of charging based on kWh needed and time of day",
        parameterSchema: [
            "type": "object",
            "properties": [
                "kwh": ["type": "number", "description": "Energy needed in kWh"],
                "hour": ["type": "integer", "description": "Hour of day (0-23)"]
            ],
            "required": ["kwh", "hour"]
        ],
        execute: { args in
            let kwh = (args["kwh"] as? Double) ?? Double(args["kwh"] as? Int ?? 0)
            let hour = (args["hour"] as? Int) ?? Int(args["hour"] as? Double ?? 0)
            let tariff = getTariff(hour: hour)
            let cost = kwh * tariff
            let period: String
            switch hour {
            case 0..<6: period = "off-peak"
            case 17..<21: period = "peak"
            default: period = "normal"
            }
            return "Charging \(String(format: "%.1f", kwh)) kWh at \(String(format: "%02d", hour)):00 (\(period) tariff: \(String(format: "%.2f", tariff)) DKK/kWh) = \(String(format: "%.2f", cost)) DKK"
        }
    )
]
```

</details>

### Verificer din løsning

Kør cellen nedenfor for at teste dine Execute-closures **uden LLM**:

In [ ]:
// Test dine tools direkte!
print("=== Test get_station_status ===")
print(tools["get_station_status"]!.execute(["stationId": "CPH-001"]))
print()

print("=== Test list_stations ===")
print(tools["list_stations"]!.execute([:])) 
print()

print("=== Test calculate_charging_cost ===")
print(tools["calculate_charging_cost"]!.execute(["kwh": 50.0, "hour": 18]))
print()

print("✅ Alle tests kørte! Hvis du ser output ovenfor, virker dine tools.")

## 1.5 The Agentic Loop - det er det hele

Nu bygger vi det komplette agentic loop fra bunden med URLSession og JSONSerialization.

In [ ]:
import Foundation

// Hjælpefunktion: Konverter SimpleTool til Ollama's tool format
func toOllamaFormat(_ tool: SimpleTool) -> [String: Any] {
    return [
        "type": "function",
        "function": [
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.parameterSchema
        ] as [String: Any]
    ]
}

// HTTP POST til Ollama
func ollamaChat(messages: [[String: Any]], tools: [[String: Any]]) async -> [String: Any]? {
    let url = URL(string: "http://localhost:11434/api/chat")!
    var request = URLRequest(url: url)
    request.httpMethod = "POST"
    request.setValue("application/json", forHTTPHeaderField: "Content-Type")
    request.timeoutInterval = 120

    let body: [String: Any] = [
        "model": selectedModelName,
        "messages": messages,
        "tools": tools,
        "stream": false
    ]

    guard let httpBody = try? JSONSerialization.data(withJSONObject: body) else { return nil }
    request.httpBody = httpBody

    do {
        let (data, _) = try await URLSession.shared.data(for: request)
        return try JSONSerialization.jsonObject(with: data) as? [String: Any]
    } catch {
        print("Error: \(error)")
        return nil
    }
}

print("✅ ollamaChat helper defined")

In [ ]:
// THE AGENTIC LOOP - no magic!
func runAgent(userMessage: String, tools: [String: SimpleTool]) async -> String {
    var messages: [[String: Any]] = [
        ["role": "system", "content": """
            Du er en hjælpsom EV-ladnings assistent for ChargeSmart netværket i København.
            Brug de tilgængelige tools til at besvare spørgsmål om ladestationer,
            tilgængelighed, og priser. Svar altid på dansk.
            """],
        ["role": "user", "content": userMessage]
    ]

    let ollamaTools = tools.values.map { toOllamaFormat($0) }
    let maxIterations = 10

    for iteration in 1...maxIterations {
        print("\n--- Iteration \(iteration) ---")

        // 1. Send til Ollama med tool-definitioner
        guard let response = await ollamaChat(messages: messages, tools: ollamaTools),
              let assistantMessage = response["message"] as? [String: Any] else {
            return "Error: No response from LLM"
        }

        messages.append(assistantMessage)

        // 2. Tjek for tool calls
        guard let toolCalls = assistantMessage["tool_calls"] as? [[String: Any]], !toolCalls.isEmpty else {
            // Ingen tool calls = LLM er færdig
            print("✅ LLM finished (no tool calls)")
            return assistantMessage["content"] as? String ?? "No response"
        }

        // 3. Eksekvér hver tool call SELV
        print("🔧 LLM wants to call \(toolCalls.count) tool(s)")

        for toolCall in toolCalls {
            guard let function = toolCall["function"] as? [String: Any],
                  let toolName = function["name"] as? String else { continue }
            let toolArgs = function["arguments"] as? [String: Any] ?? [:]
            print("   Tool: \(toolName)")
            print("   Args: \(toolArgs)")

            // Find og kør vores SimpleTool
            if let tool = tools[toolName] {
                let result = tool.execute(toolArgs)  // <-- DIN kode kører her!
                let preview = String(result.prefix(80))
                print("   📤 Result: \(preview)...")

                // 4. Tilføj resultat til messages
                messages.append(["role": "tool", "content": result])
            } else {
                messages.append(["role": "tool", "content": "Error: Unknown tool \(toolName)"])
            }
        }
        // Loop fortsætter - LLM ser resultaterne
    }

    return "Max iterations reached"
}

print("✅ runAgent function defined")

## 1.6 Øvelse 2: Human-in-the-Loop Confirmation
*~15 minutter*

I produktion vil du ofte have tools der gør "farlige" ting - f.eks. starter en ladesession, foretager en betaling, eller sletter data. Før sådanne tools eksekveres, bør agenten **bede om bekræftelse**.

**Scenarie:** Forestil dig at `calculate_charging_cost` kunne udløse en forhåndsreservation eller pre-authorization på dit kort. Før vi eksekverer det, vil vi gerne have brugerens godkendelse.

**Din opgave:** Modificer loopet så det beder om bekræftelse før "farlige" tools eksekveres.

In [ ]:
// 🎯 ØVELSE 2: Tilføj human-in-the-loop confirmation

import Foundation

// Definer hvilke tools der kræver bekræftelse
let dangerousTools: Set<String> = ["calculate_charging_cost"]

// RunAgentWithConfirmation - næsten identisk med runAgent!
// Bruger readLine() til at spørge brugeren
func runAgentWithConfirmation(
    userMessage: String,
    tools: [String: SimpleTool]
) async -> String {
    var messages: [[String: Any]] = [
        ["role": "system", "content": """
            Du er en hjælpsom EV-ladnings assistent for ChargeSmart netværket i København.
            Brug de tilgængelige tools til at besvare spørgsmål om ladestationer,
            tilgængelighed, og priser. Svar altid på dansk.
            """],
        ["role": "user", "content": userMessage]
    ]

    let ollamaTools = tools.values.map { toOllamaFormat($0) }
    let maxIterations = 10

    for iteration in 1...maxIterations {
        print("\n--- Iteration \(iteration) ---")

        guard let response = await ollamaChat(messages: messages, tools: ollamaTools),
              let assistantMessage = response["message"] as? [String: Any] else {
            return "Error: No response from LLM"
        }

        messages.append(assistantMessage)

        guard let toolCalls = assistantMessage["tool_calls"] as? [[String: Any]], !toolCalls.isEmpty else {
            print("✅ LLM finished (no tool calls)")
            return assistantMessage["content"] as? String ?? "No response"
        }

        print("🔧 LLM wants to call \(toolCalls.count) tool(s)")

        for toolCall in toolCalls {
            guard let function = toolCall["function"] as? [String: Any],
                  let toolName = function["name"] as? String else { continue }
            let toolArgs = function["arguments"] as? [String: Any] ?? [:]
            print("   Tool: \(toolName)")
            print("   Args: \(toolArgs)")

            // 🎯 ØVELSE: Bekræftelseslogik!
            if dangerousTools.contains(toolName) {
                print("⚠️ Execute '\(toolName)'? (y/n): ", terminator: "")
                let confirmation = readLine()
                if confirmation?.trimmingCharacters(in: .whitespaces).lowercased() != "y" {
                    print("   ⏭️ Skipped by user")
                    messages.append(["role": "tool", "content": "Tool \(toolName) was skipped by user"])
                    continue
                }
                print("   ✅ Confirmed by user")
            }

            if let tool = tools[toolName] {
                let result = tool.execute(toolArgs)
                let preview = String(result.prefix(80))
                print("   📤 Result: \(preview)...")
                messages.append(["role": "tool", "content": result])
            } else {
                messages.append(["role": "tool", "content": "Error: Unknown tool \(toolName)"])
            }
        }
    }

    return "Max iterations reached"
}

print("✅ runAgentWithConfirmation defined")

### Om readLine()

`readLine()` er Swift's måde at få bruger-input. I en Jupyter Notebook kan den åbne en input-dialog.

<details>
<summary>💡 Reference: Bekræftelseslogikken</summary>

```swift
// Tjek om tool er "farligt" og bed om bekræftelse
if dangerousTools.contains(toolName) {
    print("⚠️ Execute '\(toolName)'? (y/n): ", terminator: "")
    let confirmation = readLine()
    if confirmation?.trimmingCharacters(in: .whitespaces).lowercased() != "y" {
        print("   ⏭️ Skipped by user")
        messages.append(["role": "tool", "content": "Tool \(toolName) was skipped by user"])
        continue  // Skip til næste tool call
    }
    print("   ✅ Confirmed by user")
}
```

**Lærdom:** Human-in-the-loop er kritisk for produktions-agents. Du bestemmer hvilke handlinger der kræver godkendelse!

</details>

### Test din løsning

Kør cellen nedenfor. Når LLM vil kalde `calculate_charging_cost`, vil du blive bedt om input. Skriv `y` for at godkende eller `n` for at afvise.

In [ ]:
// Test human-in-the-loop
let confirmResult = await runAgentWithConfirmation(
    userMessage: "Hvad koster det at lade 50 kWh kl. 18?",
    tools: tools
)

print("\n=== Final Response ===")
print(confirmResult)

## 1.7 Test din Agent!

In [ ]:
let result1 = await runAgent(
    userMessage: "Hvilke ladestationer er ledige lige nu?",
    tools: tools
)

print("\n=== Final Response ===")
print(result1)

In [ ]:
let result2 = await runAgent(
    userMessage: "Hvad er status på CPH-002?",
    tools: tools
)

print("\n=== Final Response ===")
print(result2)

In [ ]:
let result3 = await runAgent(
    userMessage: "Hvad koster det at lade 50 kWh kl. 18? Og hvad hvis jeg venter til kl. 22?",
    tools: tools
)

print("\n=== Final Response ===")
print(result3)

## Checkpoint Del 1

Du har nu:
1. ✅ Set præcis hvad LLM'en modtager (JSON tool definitions)
2. ✅ Set præcis hvad LLM'en returnerer (JSON tool calls)
3. ✅ Bygget `SimpleTool` - en transparent tool abstraction
4. ✅ **Øvelse 1:** Skrevet Execute-closures til 3 ChargeSmart tools
5. ✅ Set det komplette agentic loop med URLSession!
6. ✅ **Øvelse 2:** Tilføjet human-in-the-loop confirmation til loopet
7. ✅ Kørt en agent med dine tools

**Nøgleindsigt:** Det er så simpelt. Et loop + nogle linjer per tool. *The emperor has no clothes!*

**Lærdom fra Øvelse 2:** I produktion er human-in-the-loop kritisk. Du bestemmer hvilke handlinger der kræver godkendelse!

---

# Del 2: Abstraktionsstigen
*15 minutter (valgfri)*

Nu hvor du forstår hvad der sker "under the hood", lad os se hvordan man kan pakke det samme ind i protokoller.

## 2.1 Side-by-Side: SimpleTool vs Protocol-based Tool

In [ ]:
// DIN SimpleTool - alt er synligt:
let simpleToolExample = SimpleTool(
    name: "get_station_status",
    description: "Get EV charging station status",
    parameterSchema: [
        "type": "object",
        "properties": ["stationId": ["type": "string"]],
        "required": ["stationId"]
    ],
    execute: { args in
        "Station \(args["stationId"] ?? "?") is available"  // <-- Koden er HER
    }
)

// Ollama's JSON format - samme data, anden struktur:
let ollamaToolExample: [String: Any] = [
    "type": "function",
    "function": [
        "name": "get_station_status",
        "description": "Get EV charging station status",
        "parameters": [
            "type": "object",
            "properties": [
                "stationId": ["type": "string", "description": "Station ID"]
            ],
            "required": ["stationId"]
        ] as [String: Any]
    ] as [String: Any]
]
// BEMÆRK: Ingen execute closure! Du skal håndtere eksekveringen separat i dit loop.

print("SimpleTool har:")
print("  - name, description, parameterSchema")
print("  - execute: ([String: Any]) -> String <- KODEN ER HER")
print()
print("Ollama's JSON tool har:")
print("  - type, function (name, description, parameters)")
print("  - Ingen execute! <- Du skal selv matche tool calls til kode")

## 2.2 Nøgleindsigt

**Ollama's tool JSON er bare en data-struktur til JSON-serialisering.**

Det er derfor vi byggede `SimpleTool` først - for at se det fulde billede:
1. Tool *definition* (JSON schema) → sendes til LLM
2. Tool *execution* (din kode) → kører lokalt når LLM requester det

Ollama's API giver dig #1, men du skal selv håndtere #2.
Vores `SimpleTool` pakker begge dele sammen så det er tydeligt.

---

# ☕ PAUSE (15 min)

---

# Del 3: Abstraktionsstigen – MCP
*45 minutter inkl. øvelse*

Vi fortsætter op ad abstraktionsstigen: SimpleTool → Ollama JSON Tool → **MCP**.

## Teori: Model Context Protocol (MCP)

### Problemet: N×M Integrationer

<img src="images/mcp-problem.svg" alt="MCP Problem" width="600" style="max-width: 100%;">

Uden standardisering: Hver agent skal have custom integration til hver tool.

### Løsningen: MCP som Standard

<img src="images/mcp-solution.svg" alt="MCP Solution" width="600" style="max-width: 100%;">

**MCP = USB-C for AI tools.** *Plug any tool into any agent.*

## 3.1 Hvad Ændrer Sig med MCP?

**Før (Del 1-2):** Tools defineret inline i din kode
```swift
var tools: [String: SimpleTool] = [
    "get_station_status": SimpleTool(..., execute: { args in ... })
]
// Tools lever i din agent kode
```

**Efter (Del 3):** Tools kommer fra eksterne MCP servere
```swift
let mcpTools = try await mcpClient.listTools()
// Tools opdages dynamisk fra server!
```

**Samme agent loop, men tools er *"pluggable"*.**

## 3.2 MCP Protocol Basics

MCP bruger **JSON-RPC 2.0** over stdio (lokalt) eller SSE (remote).

Tre hovedoperationer:
1. `initialize` - Handshake mellem client og server
2. `tools/list` - Hent liste af tilgængelige tools
3. `tools/call` - Kald et specifikt tool

## 3.3 Øvelse 3: Protocol-based Tools med Swift Protocols
*~10 minutter*

MCP er designet til **inter-proces** kommunikation (client og server i separate processer).
I en notebook bruger vi i stedet Swift protocols, som giver samme tool-oplevelse **in-process**.

> **Note:** I produktion ville du forbinde til en ekstern MCP server via stdio eller HTTP transport.

**📋 Opgave:** Sammenlign `AITool` protocol med `SimpleTool`:
- I Øvelse 1 brugte du `SimpleTool` med `execute: { args in ... }` (closure)
- Her bruger du en `struct` der conformer til `AITool` protocol
- Parametrene er de samme, men strukturen er mere organiseret!

**Bemærk:** Logikken er den samme - kun strukturen er anderledes.

In [ ]:
// 🎯 ØVELSE 3: Protocol-based tools

protocol AITool {
    var name: String { get }
    var description: String { get }
    var parameterSchema: [String: Any] { get }
    func execute(args: [String: Any]) -> String
}

// Eksempel: get_station_status er givet
struct GetStationStatusTool: AITool {
    let name = "get_station_status"
    let description = "Get real-time status of an EV charging station including availability, power level, and current session info"
    let parameterSchema: [String: Any] = [
        "type": "object",
        "properties": ["stationId": ["type": "string", "description": "Station ID like CPH-001"]],
        "required": ["stationId"]
    ]

    func execute(args: [String: Any]) -> String {
        guard let stationId = args["stationId"] as? String else { return "Error: Missing stationId" }
        guard let station = stations[stationId] else { return "Error: Station \(stationId) not found" }
        if let session = activeSessions[stationId] {
            let formatter = DateFormatter()
            formatter.dateFormat = "HH:mm"
            return "Station \(stationId) (\(station.name)): IN USE by \(session.vehicleId) since \(formatter.string(from: session.start)), \(String(format: "%.1f", session.kwh)) kWh delivered. \(station.powerKw)kW \(station.type) charger."
        }
        return "Station \(stationId) (\(station.name)): AVAILABLE. \(station.powerKw)kW \(station.type) charger at \(station.location)."
    }
}

struct ListStationsTool: AITool {
    let name = "list_stations"
    let description = "List all EV charging stations in the ChargeSmart network with their current availability"
    let parameterSchema: [String: Any] = ["type": "object", "properties": [:] as [String: Any]]

    func execute(args: [String: Any]) -> String {
        let lines = stations.values.sorted(by: { $0.id < $1.id }).map { s in
            let status = activeSessions.keys.contains(s.id) ? "IN USE" : "AVAILABLE"
            return "- \(s.id): \(s.name) (\(s.location)) - \(s.powerKw)kW \(s.type) - \(status)"
        }
        return "ChargeSmart Network Stations:\n" + lines.joined(separator: "\n")
    }
}

struct CalculateChargingCostTool: AITool {
    let name = "calculate_charging_cost"
    let description = "Calculate the cost of charging based on kWh needed and hour of day (0-23)"
    let parameterSchema: [String: Any] = [
        "type": "object",
        "properties": [
            "kwh": ["type": "number", "description": "Energy needed in kWh"],
            "hour": ["type": "integer", "description": "Hour of day (0-23)"]
        ],
        "required": ["kwh", "hour"]
    ]

    func execute(args: [String: Any]) -> String {
        let kwh: Double
        if let v = args["kwh"] as? Double { kwh = v }
        else if let v = args["kwh"] as? Int { kwh = Double(v) }
        else if let v = args["kwh"] as? String, let p = Double(v) { kwh = p }
        else { return "Error: Invalid kwh" }

        let hour: Int
        if let v = args["hour"] as? Int { hour = v }
        else if let v = args["hour"] as? Double { hour = Int(v) }
        else if let v = args["hour"] as? String, let p = Int(v) { hour = p }
        else { return "Error: Invalid hour" }

        let tariff = getTariff(hour: hour)
        let cost = kwh * tariff
        let period: String
        switch hour {
        case 0..<6: period = "off-peak"
        case 17..<21: period = "peak"
        default: period = "normal"
        }
        return "Charging \(String(format: "%.1f", kwh)) kWh at \(String(format: "%02d", hour)):00 (\(period) tariff: \(String(format: "%.2f", tariff)) DKK/kWh) = \(String(format: "%.2f", cost)) DKK"
    }
}

let aiTools: [AITool] = [
    GetStationStatusTool(),
    ListStationsTool(),
    CalculateChargingCostTool()
]

print("🔌 Oprettet \(aiTools.count) AITool protocol-based tools:")
for t in aiTools {
    print("   - \(t.name): \(String(t.description.prefix(50)))...")
}

### Hints til Øvelse 3

<details>
<summary>💡 Hint: ListStationsTool</summary>

```swift
struct ListStationsTool: AITool {
    let name = "list_stations"
    let description = "List all EV charging stations..."
    let parameterSchema: [String: Any] = ["type": "object", "properties": [:] as [String: Any]]

    func execute(args: [String: Any]) -> String {
        let lines = stations.values.sorted(by: { $0.id < $1.id }).map { s in
            let status = activeSessions.keys.contains(s.id) ? "IN USE" : "AVAILABLE"
            return "- \(s.id): \(s.name) (\(s.location)) - \(s.powerKw)kW \(s.type) - \(status)"
        }
        return "ChargeSmart Network Stations:\n" + lines.joined(separator: "\n")
    }
}
```

</details>

<details>
<summary>💡 Hint: CalculateChargingCostTool</summary>

```swift
struct CalculateChargingCostTool: AITool {
    let name = "calculate_charging_cost"
    // ...
    func execute(args: [String: Any]) -> String {
        let kwh = (args["kwh"] as? Double) ?? Double(args["kwh"] as? Int ?? 0)
        let hour = (args["hour"] as? Int) ?? Int(args["hour"] as? Double ?? 0)
        let tariff = getTariff(hour: hour)
        let cost = kwh * tariff
        // ... format result
    }
}
```

</details>

## 3.4 Automatiseret Tool Calling med AITool Protocol

Her kommer magien - eller rettere, **automationen af det loop du byggede i Del 1!**

Vi skriver en hjælpefunktion der bruger `AITool` protocol til automatisk at dispatche tool calls.

In [ ]:
// Automatiseret agent med AITool protocol
func runAgentWithAITools(userMessage: String, aiTools: [AITool]) async -> String {
    var messages: [[String: Any]] = [
        ["role": "system", "content": """
            Du er en hjælpsom EV-ladnings assistent for ChargeSmart netværket i København.
            Brug de tilgængelige tools til at besvare spørgsmål om ladestationer,
            tilgængelighed, og priser. Svar altid på dansk.
            """],
        ["role": "user", "content": userMessage]
    ]

    // Konverter AITool til Ollama format
    let ollamaTools: [[String: Any]] = aiTools.map { tool in
        [
            "type": "function",
            "function": [
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.parameterSchema
            ] as [String: Any]
        ]
    }

    // Opret dictionary til hurtig lookup
    var toolMap: [String: AITool] = [:]
    for tool in aiTools {
        toolMap[tool.name] = tool
    }

    let maxIterations = 10
    for iteration in 1...maxIterations {
        print("\n--- Iteration \(iteration) ---")

        guard let response = await ollamaChat(messages: messages, tools: ollamaTools),
              let assistantMessage = response["message"] as? [String: Any] else {
            return "Error: No response from LLM"
        }

        messages.append(assistantMessage)

        guard let toolCalls = assistantMessage["tool_calls"] as? [[String: Any]], !toolCalls.isEmpty else {
            print("✅ LLM finished (no tool calls)")
            return assistantMessage["content"] as? String ?? "No response"
        }

        print("🔧 LLM wants to call \(toolCalls.count) tool(s)")

        for toolCall in toolCalls {
            guard let function = toolCall["function"] as? [String: Any],
                  let toolName = function["name"] as? String else { continue }
            let toolArgs = function["arguments"] as? [String: Any] ?? [:]

            // Automatisk dispatch via protocol!
            if let tool = toolMap[toolName] {
                let result = tool.execute(args: toolArgs)
                print("   📤 \(toolName) -> \(String(result.prefix(60)))...")
                messages.append(["role": "tool", "content": result])
            } else {
                messages.append(["role": "tool", "content": "Error: Unknown tool \(toolName)"])
            }
        }
    }
    return "Max iterations reached"
}

print("✅ runAgentWithAITools defined")

## 3.5 Test: ChargeSmart med AITool Protocol

Kør samme query som i Del 1, men nu med protocol-based tools og automatiseret dispatch!

**Bemærk:** Samme tools, samme data - men nu via `AITool` protocol.

In [ ]:
print("Sender besked med AITool protocol tools...\n")

let aiResult = await runAgentWithAITools(
    userMessage: "Hvilke ladestationer er ledige lige nu? Og hvad koster det at lade 30 kWh kl. 18?",
    aiTools: aiTools
)

print("\n=== Final Response ===")
print(aiResult)
print("\n✅ Done!")

## Checkpoint Del 3

Du har nu:
1. ✅ Forstået MCP som "USB-C for AI tools"
2. ✅ **Øvelse 3:** Oprettet tools med Swift `AITool` protocol
3. ✅ Set hvordan automatiseret tool dispatch virker
4. ✅ Kørt en agent med protocol-based tool invocation

**Nøgleindsigt:** MCP standardiserer tool connectivity. I produktion forbinder du til eksterne MCP servere - i notebooks bruger vi protocols som in-process alternativ.

---

# Del 3.5: Øvelse 4 - Udvid med get_nearest_station
*~20 minutter*

Nu skal du **studere hvordan man udvider** agenten med et helt nyt tool! Dette er den vigtigste øvelse, fordi det simulerer det typiske workflow: du har en fungerende agent og skal tilføje ny funktionalitet.

## Scenarie

En bruger står på gaden og vil finde den **nærmeste ledige ladestation**. De kender deres GPS-koordinater og vil have en anbefaling.

**📋 Opgave:** Studer implementeringen nedenfor og forstå logikken:
1. Modtag brugerens latitude og longitude
2. Filtrer alle **ledige** stationer (ikke optaget)
3. Beregn afstanden til hver station med Haversine
4. Returner den nærmeste

## Stationskoordinater

Alle stationer har nu latitude/longitude (vi tilføjede dem i starten):

| Station | Lokation | Latitude | Longitude |
|---------|----------|----------|-----------|
| CPH-001 | Nørreport | 55.6839 | 12.5715 |
| CPH-002 | Fisketorvet | 55.6692 | 12.5519 |
| CPH-003 | Tivoli | 55.6736 | 12.5681 |
| CPH-004 | Ørestad | 55.6310 | 12.5770 |
| CPH-005 | Amager | 55.6520 | 12.6050 |
| CPH-006 | Nørrebro | 55.7015 | 12.5450 |
| CPH-007 | Frederiksberg | 55.6784 | 12.5293 |
| CPH-008 | Kastrup | 55.6180 | 12.6560 |
| CPH-009 | Valby | 55.6631 | 12.5120 |
| CPH-010 | Hellerup | 55.7280 | 12.5720 |

In [ ]:
import Foundation

// Haversine-formlen til at beregne afstand mellem to koordinater (i km)
func calculateDistance(lat1: Double, lon1: Double, lat2: Double, lon2: Double) -> Double {
    let R = 6371.0 // Jordens radius i km
    let dLat = (lat2 - lat1) * .pi / 180
    let dLon = (lon2 - lon1) * .pi / 180
    let a = sin(dLat / 2) * sin(dLat / 2) +
            cos(lat1 * .pi / 180) * cos(lat2 * .pi / 180) *
            sin(dLon / 2) * sin(dLon / 2)
    let c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c
}

// Test: Afstand fra Rådhuspladsen (55.6761, 12.5683) til Nørreport (CPH-001)
let testDistance = calculateDistance(lat1: 55.6761, lon1: 12.5683, lat2: 55.6839, lon2: 12.5715)
print("Test: Afstand fra Rådhuspladsen til Nørreport: \(String(format: "%.2f", testDistance)) km")

In [ ]:
// 🎯 DIN TUR: Implementer get_nearest_station som AITool

struct GetNearestStationTool: AITool {
    let name = "get_nearest_station"
    let description = "Find the nearest available charging station. Parameters: latitude and longitude of user's current position"
    let parameterSchema: [String: Any] = [
        "type": "object",
        "properties": [
            "latitude": ["type": "number", "description": "User's latitude"],
            "longitude": ["type": "number", "description": "User's longitude"]
        ],
        "required": ["latitude", "longitude"]
    ]

    func execute(args: [String: Any]) -> String {
        // TODO: 1. Parse latitude og longitude fra args
        //       2. Filtrer ledige stationer fra stations.values
        //       3. Find nærmeste med calculateDistance()
        //       4. Returner beskrivende streng

        fatalError("Implementer get_nearest_station!")
    }
}

// Tilføj til aiTools
var extendedAiTools: [AITool] = aiTools + [GetNearestStationTool()]
print("🔌 extendedAiTools har nu \(extendedAiTools.count) tools (inkl. get_nearest_station stub)")

### Hints til Øvelse 4

<details>
<summary>💡 Hint 1: Filtrer ledige stationer</summary>

```swift
let availableStations = stations.values.filter { !activeSessions.keys.contains($0.id) }

if availableStations.isEmpty {
    return "Ingen ledige stationer lige nu!"
}
```

</details>

<details>
<summary>💡 Hint 2: Beregn afstande og find minimum</summary>

```swift
let withDistances = availableStations.map { s in
    (station: s, distance: calculateDistance(lat1: latitude, lon1: longitude, lat2: s.latitude, lon2: s.longitude))
}
let nearest = withDistances.min(by: { $0.distance < $1.distance })!
```

</details>

<details>
<summary>✅ Fuld løsning</summary>

```swift
struct GetNearestStationTool: AITool {
    let name = "get_nearest_station"
    let description = "Find the nearest available charging station. Parameters: latitude and longitude of user's current position"
    let parameterSchema: [String: Any] = [
        "type": "object",
        "properties": [
            "latitude": ["type": "number", "description": "User's latitude"],
            "longitude": ["type": "number", "description": "User's longitude"]
        ],
        "required": ["latitude", "longitude"]
    ]

    func execute(args: [String: Any]) -> String {
        let latitude: Double
        if let v = args["latitude"] as? Double { latitude = v }
        else if let v = args["latitude"] as? String, let p = Double(v) { latitude = p }
        else { return "Error: Invalid latitude" }

        let longitude: Double
        if let v = args["longitude"] as? Double { longitude = v }
        else if let v = args["longitude"] as? String, let p = Double(v) { longitude = p }
        else { return "Error: Invalid longitude" }

        let availableStations = stations.values.filter { !activeSessions.keys.contains($0.id) }
        if availableStations.isEmpty { return "Ingen ledige stationer lige nu!" }

        let withDistances = availableStations.map { s in
            (station: s, distance: calculateDistance(lat1: latitude, lon1: longitude, lat2: s.latitude, lon2: s.longitude))
        }
        let nearest = withDistances.min(by: { $0.distance < $1.distance })!

        return "Nærmeste ledige station: \(nearest.station.id) (\(nearest.station.name)) - " +
               "\(String(format: "%.2f", nearest.distance)) km væk. " +
               "\(nearest.station.powerKw)kW \(nearest.station.type) charger ved \(nearest.station.location)."
    }
}
```

</details>

### Test dit nye tool

In [ ]:
// Test get_nearest_station med en position nær Rådhuspladsen
print("=== Test fra Rådhuspladsen (55.6761, 12.5683) ===\n")

let nearestResult = await runAgentWithAITools(
    userMessage: "Jeg står på Rådhuspladsen (koordinater: 55.6761, 12.5683). Hvor er den nærmeste ledige ladestation?",
    aiTools: extendedAiTools
)

print("\n=== Final Response ===")
print(nearestResult)
print("\n✅ Test færdig!")

## Checkpoint Øvelse 4

Du har nu:
1. ✅ Forstået Haversine-formlen til afstandsberegning
2. ✅ Implementeret `get_nearest_station` med filter og afstandssortering
3. ✅ Udvidet en eksisterende agent med ny funktionalitet
4. ✅ Testet med en realistisk brugerforespørgsel

**Lærdom:** At udvide en agent er ofte nemmere end at bygge en fra bunden. Du tilføjer bare et nyt tool til listen!

---

# Del 4: Skills – teori og prøv med GitHub Copilot
*25 minutter*

## Fra domæne-specifikke agents til universel kode

**Hvordan vi plejede at tænke:**

<img src="images/anthropic-how-we-used-to-think-about-agents.png" alt="Separate agents for each domain" width="600" style="max-width: 100%;">

Vi troede hver domæne kræver sin egen agent med egne tools og scaffolding.

**Hvad vi opdagede:**

<img src="images/anthropic-code-is-the-universal-interface.png" alt="Code is the universal interface" width="600" style="max-width: 100%;">

Kode er ikke bare en use case – det er den universelle grænseflade til den digitale verden.

En coding agent kan:
- Kalde API'er for at hente data (research)
- Organisere data i filsystemet (dokumenter)
- Analysere med Python (finans)
- Generere output i ethvert format (marketing)

**Konklusion**: Vi behøver ikke mange forskellige agents – vi behøver ÉN general-purpose coding agent + **skills** der giver domæneekspertise.

## Teori: Hvorfor Skills?

**Problemet:** Giv en model 100 tools, og den bliver forvirret. Tool selection degraderer.

**Løsningen:** Skills = "lazy-loaded expertise"
- En "router" klassificerer først brugerens intent
- Kun relevante tools/knowledge indlæses
- Modellen fokuserer på én opgave ad gangen

**Key insight:** Copilot har allerede loopet, LLM, og basale tools. Du tilføjer domæneviden!

## 4.1 GitHub Copilot Agent Mode

VS Code 1.108+ har eksperimentel support for "agent skills".

### Aktivering

1. Åbn VS Code Settings (Ctrl+,)
2. Søg efter `chat.useAgentSkills`
3. Aktiver indstillingen

### Skill Struktur

```
.github/skills/
└── ev-charging-advisor/
    └── SKILL.md
```

Copilot scanner automatisk `.github/skills/` og indlæser relevante skills.

## 4.2 SKILL.md Anatomi

```yaml
---
name: my-skill
description: Kort beskrivelse til discovery
triggers:
  - "når bruger spørger om X"
  - "når der er behov for Y"
---

# Detaljerede Instruktioner

Her kommer den fulde domæneviden...
```

- **YAML frontmatter:** Metadata til skill discovery
- **Markdown body:** Instruktioner, eksempler, context

## 4.3 Øvelse: Opret EV Charging Advisor Skill

### Step 1: Opret skill mappe

**PowerShell:**
```powershell
New-Item -ItemType Directory -Force -Path ".github/skills/ev-charging-advisor"
```

**Bash:**
```bash
mkdir -p .github/skills/ev-charging-advisor
```

### Step 2: Opret SKILL.md

Opret filen `.github/skills/ev-charging-advisor/SKILL.md`:

```yaml
---
name: ev-charging-advisor
description: Ekspertrådgivning om optimering af elbil-ladning på ChargeSmart netværket
triggers:
  - "når bruger spørger om ladepris"
  - "når bruger vil finde billigste ladetidspunkt"
  - "når bruger spørger om batteri-tips"
  - "når bruger nævner ChargeSmart eller elbil-ladning"
---

# EV Charging Advisor

Du er en ekspert i at optimere elbil-ladning på ChargeSmart netværket i København.

## ChargeSmart Tarif-struktur

| Periode | Tid | Pris/kWh |
|---------|-----|----------|
| Off-peak | 00:00-06:00 | 1.50 DKK |
| Normal | 06:00-17:00, 21:00-00:00 | 2.50 DKK |
| Peak | 17:00-21:00 | 4.00 DKK |

## Stationer i Netværket

- CPH-001: Nørreport Station - 150kW ultra-fast
- CPH-002: Fisketorvet Parking - 50kW fast
- CPH-003: Tivoli Garage - 50kW fast
- CPH-004: Ørestad Center - 150kW ultra-fast
- CPH-005: Amager Strandpark - 22kW slow
- CPH-006: Nørrebro Runddel - 50kW fast
- CPH-007: Frederiksberg Have - 7kW slow
- CPH-008: Kastrup Lufthavn P4 - 150kW ultra-fast
- CPH-009: Valby Langgade - 22kW slow
- CPH-010: Hellerup Station - 50kW fast

## Instruktioner

Når bruger spørger om ladning:

1. **Identificer behov** - Hvor mange kWh skal de lade?
2. **Tjek tidspunkt** - Hvornår vil de lade?
3. **Beregn pris** - Brug tarif-tabellen
4. **Optimer** - Foreslå ALTID billigere alternativer
5. **Batteri-tips** - Tilføj relevante tips

## Batteri-tips

- Undgå at lade til 100% dagligt (80% er optimalt for battery health)
- Undgå at køre under 20% regelmæssigt
- Forvarm batteri i koldt vejr før hurtigladning
- Ultra-fast ladning (150kW) slider mere på batteriet end langsom ladning

## Eksempel-dialog

**Bruger:** Hvad koster det at lade 40 kWh kl. 18?

**Svar:** At lade 40 kWh kl. 18:00 koster **160 DKK** (peak-tarif: 4.00 DKK/kWh).

💡 **Tip:** Hvis du kan vente til kl. 21:00, falder prisen til **100 DKK** (normal-tarif).
Eller endnu bedre - lad natten over (kl. 00-06) for kun **60 DKK**!

🔋 **Batteri-tip:** Hvis du ikke behøver fuld opladning, så lad kun til 80% for at forlænge batteriets levetid.
```

### Step 3: Test med Copilot Chat

1. Åbn **Copilot Chat** i VS Code (Ctrl+Alt+I)
2. Prøv disse prompts:
   - "Hvad koster det at lade 50 kWh kl. 18?"
   - "Find den billigste tid at lade min elbil"
   - "Giv mig tips til at forlænge mit EV batteri"

Copilot burde nu bruge din skill til at give ChargeSmart-specifikke svar! (Det er tilfælded hvis du ser beskeden "Read SKILL.md file".)

## Checkpoint Del 4

Du har nu:
1. ✅ Forstået Skills som "lazy-loaded expertise"
2. ✅ Oprettet en SKILL.md med domæneviden
3. ✅ Udvidet GitHub Copilot med ChargeSmart ekspertise

**Nøgleindsigt:** Du behøver ikke bygge en agent - du kan udvide én du allerede har!

---

# Afrunding
*10 minutter*

## Key Takeaways

1. **Agents er simple** - ~200 linjer loop kode
2. **"Magien" er i LLM'en** - ikke infrastrukturen
3. **MCP standardiserer** - USB-C for AI tools
4. **Skills tilføjer viden** - uden at genopbygge

## Næste Skridt: Enterprise Agent SDK'er

Nu hvor du har bygget din egen agent-loop og forstår abstraktionerne, kan du udforske etablerede SDK'er. Disse giver dig **observability**, **multi-agent orchestration** og **enterprise compliance** ud af boksen – men bygger på præcis de samme principper, du har lært i dag.

**To tilgange dominerer:**

| | **Anthropic Claude Agent SDK** | **Microsoft Agent Framework** |
|---|---|---|
| **Filosofi** | "Giv Claude en computer" | "Unified multi-agent orchestration" |
| **Kerne** | Bash/fil-tools, subagents, MCP | AutoGen + Semantic Kernel |
| **Sprog** | Python | Python, C#, Java |
| **Styrke** | Simpel agent → computer-interaktion | Enterprise: observability, compliance, A2A |
| **MCP** | ✅ Native integration | ✅ Native integration |

![Agent SDK Landscape](images/agent-sdk-landscape.svg)

> **Nøgleindsigt:** Begge SDK'er bygger på de samme principper, du har lært (tools, loops, MCP). Forskellen ligger i *hvor meget* infrastruktur du får "gratis" – og *hvilken leverandør* du binder dig til.

## The Emperor Has No Clothes

> **"The core of these tools isn't magic. It's about 200 lines of straightforward Python."**
> — Mihail Eric

Du har nu set det selv. Du kan bygge en agent. Du forstår hvad der sker.

**Hvorfor er dette vigtigt?**

> Whilst software development/programming is now dead. We however deeply need software engineers with these skills who understand that LLMs are a new form of programmable computer. If you haven't built your own coding agent yet - please do.
> — [Geoffrey Huntley](https://ghuntley.com/loop/)

## Stof til eftertanke

Kan du komme i tanke om systemer hvor statiske formularer, der forudsætter, at brugeren på forhånd ved, hvad vedkommende har brug for, kunne erstattes af multi-modale brugergrænseflader, der forstår brugerens intention?

![Gammel vs. Ny UX-Logik](images/gammel-vs-ny-logik-ux.png)

## Ressourcer

- [MCP Documentation](https://modelcontextprotocol.io)
- [GitHub Copilot Skills](https://docs.github.com/en/copilot)
- [The Emperor Has No Clothes](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)
- [Building agents with the Claude Agent SDK](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)
- [Introducing Microsoft Agent Framework](https://azure.microsoft.com/en-us/blog/introducing-microsoft-agent-framework/)

---